In [ ]:
# Since some cells will fail on purpose in this notebook, only report minimal (meaningful) error messages
%xmode minimal

from dataclasses import dataclass, field, replace
import time

import jax
import equinox as eqx
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# Pytrees

First mentioned in the introductory BasicJAX notebook: JAX transformations (`jit`, `vmap`, ...) produce functions whose inputs and outputs are pytrees.
A pytree is a nested structure built from container types (like lists, tuples, and dicts) whose contents are themselves pytrees or leaves.
A leaf is anything that isn't one of those containers, such as a JAX array, a NumPy array, or a Python scalar.

An example of pytree:

In [ ]:
A = (4., {'a': jnp.array([1., 2., 3.]), 'b': (45, 2, 6.)})

[The `jax.tree` submodule exposes many functions to manipulate pytrees.](https://docs.jax.dev/en/latest/jax.tree.html)

A pytree can also be seen as a flattened list of its leaves combined with a treedef, which basically is its blueprint.
To manipulate them:
- get leaves of a pytree: `jax.tree.leaves`
- get treedef of a pytree: `jax.tree.structure`
- get both leaves and treedef: `jax.tree.flatten`
- build a pytree from leaves tuple and treedef: `jax.tree.unflatten`


In [ ]:
print(jax.tree.leaves(A)) # Print a flattened view of the leaves of A

print(jax.tree.structure(A)) # Print the treedef of A

leaves, treedef = jax.tree.flatten(A) # Retrieve flattened view and treedef of A at once
print(leaves)
print(treedef)

# Rebuild a pytree from A's treedef and new leaves
B = jax.tree.unflatten(treedef, (6., jnp.array([9., 7.]), 9, 5, 7.))
print(B)

The concept of pytree flattening and unflattening is quite central, and will be revisited once we tackle custom pytree (registering classes to be manipulated as pytrees).

> _**Note:**_ `dict` containers needs hashable keys, and those will be ordered during flattening. If order matters, it is best to use `collections.OrderedDict` ([link to documentation](https://docs.python.org/3/library/collections.html#collections.OrderedDict)). See [here](https://docs.jax.dev/en/latest/pytrees.html#dictionary-keys-must-be-sortable) for more details.

One can map a function which tranform leaves of a pytree, using `jax.tree.map` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.tree.map.html), read it!).

In [ ]:
# Square every elements of the leaves
print(jax.tree.map(lambda x: x**2, A))

# Sum every elements of every leaves
print(jax.tree.map(lambda x: jnp.sum(x), A))

One can reduce a pytree using `jax.tree.reduce` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.tree.reduce.html), read it!).

In [ ]:
# Sum every leaves of the tree.
print(jax.tree.reduce(lambda x, y: jnp.sum(x) + jnp.sum(y), A))

Pytree transforms are jittable:

In [ ]:
@jax.jit
def my_map(p):
    print("Compiling my_map()...")
    return jax.tree.map(lambda x: x+10., p)

print(my_map(A)) # Jitted mapping

A_ = jax.tree.map(lambda x: x**2, A)

print(my_map(A_)) # Since A_ is of the same treedef than A with same arrays size/dtype, this will not trigger recompilation

print(my_map(B)) # This however will

# Custom pytrees

## Jitting class methods
Out of the box, it is not possible to jit class methods:

In [ ]:
class A:
    def __init__(self, val, cst):
        self.val = val
        self.cst = cst

    @jax.jit
    def compute(self):
        return self.val * self.cst

a = A(jnp.array([1., 2., 3.]), 4.)
a.compute()

The problem is that the first class argument, `self`, of class `A` is an opaque leaf which JAX can't traverse.
One solution is to define it as static.
However, mutating the class will have no effect on further computations.

## Static `self`
The problem is that `self`, the first parameter of the method, is an unregistered instance: an opaque leaf JAX can't trace.
One fix is to mark it static with `static_argnums=0`.
That compiles, but at a cost: a static argument is keyed into the compilation cache by its hash, and the default object hash is identity-based.
Mutating the instance in place doesn't change its identity, so the next call is a cache hit and hands back the result compiled against the old values: the mutation is silently ignored.

In [ ]:
class A:
    def __init__(self, val1, val2, s):
        self.val1 = val1
        self.val2 = val2
        self.s = s

    @jax.jit(static_argnums=0)
    def compute(self):
        print("Compiling A.compute()...")
        return self.val1 * self.val2

a = A(jnp.array([1., 2., 3.]), 4., "hey")
print(a.compute())

a.val2 = 10.
print(a.compute()) # Should return [10. 20. 30.]
print(A(jnp.array([1., 2., 3.]), 10., "hey").compute()) # Correctly returns [10. 20. 30.] but at the price of a new compilation

## Use collection of pure functions

Another approach is to keep the class as a plain data holder and moves computations out into free-standing jitted pure functions.
Because those values now reach the compiled function as ordinary arguments, a later mutation is simply picked up on the next call:

In [ ]:
class _A:
    @staticmethod
    @jax.jit
    def compute(val1, val2):
        return val1 * val2
        
class A:
    def __init__(self, val1, val2, s):
        self.val1 = val1
        self.val2 = val2
        self.s = s

    def compute(self):
        return _A.compute(self.val1, self.val2)

a = A(jnp.array([1., 2., 3.]), 4., "hey")
print(a.compute())

a.val2 = 10.
print(a.compute())

This works, but it trades ergonomics for correctness.
Each computation now exists twice and have to stay in sync: add a field and every signature that consumes it has to change.
Mutation, that is changing the objects fields through some computation, has to be performed manually in the wrapper methods.
And last, since only the loose arrays/pytrees cross the jit boundary, you can never hand the object itself to a transformation: it won't compose with grad, vmap, or jit.

In [ ]:
@jax.jit
def f(x, a):
    return x + a.compute()

# This still won't work:
f(10., a)

## Registering pytree node
The real fix is to make the instance itself something JAX can see into: register the class as a custom pytree node, promoting it from an opaque leaf to an internal node whose array children JAX will traverse and trace.

Registration asks the class for two methods:
- `tree_flatten`: takes an instance apart into its array children plus any static metadata.
- `tree_unflatten`: rebuilds an instance from those pieces.

With that in place the whole object can cross a jit (or vmap, ...) boundary as a single argument, its arrays traced automatically and its current values read on every call.

This is commonly done through the `jax.tree_util.register_pytree_node_class` function ([link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.tree_util.register_pytree_node_class.html), read it!), which can also be used as a decorator (and easier to use that way):

In [ ]:
@jax.tree_util.register_pytree_node_class # Decorator to register our class as a node
class A:
    def __init__(self, val1, val2, s):
        self.val1 = val1
        self.val2 = val2
        self.s = s

    @classmethod
    def tree_unflatten(cls, aux_data, children):
        # Initialize an object from a tuple of auxiliary data (static metadata) and array children
        s, = aux_data
        val1, val2, = children
        return cls(val1, val2, s)

    def tree_flatten(self):
        # Transform the object into a tuple of childrens and auxiliary data.
        children = (self.val1, self.val2)
        aux_data = (self.s,)
        return (children, aux_data)

    def compute(self):
        return self.val1 * self.val2

    def __repr__(self):
        return f"A(val1={self.val1}, val2={self.val2}, cst={self.cst})"

a = A(jnp.array([1., 2., 3.]), 5, "hey")
print(a.compute())
a.val2 = 10.
print(a.compute())

Now JAX transformations are able to compose with the class:

In [ ]:
f(10., a)

Registering the class as a custom node enabled us to use it at the boundary of JAX transformations and its field values are read on every call.
However, the per-field bookkeeping didn't vanish but moved.
We now have to hand-maintain the `tree_flatten` and `tree_unflatten` where both have to be synchornized.
Adding or modifying fields means that both those methods have to be modified, and a swapped pair can silently rebuild a wrong object.

However, this bookkeeping can be easily algorithmically performed.
We first introduce a custom base class that can be used to define arbitrary nodes without having to handle these manually.
Then introduce `equinox.Module`.

## Basic PytreeDataclass

The following is an en example of a base call that can be used coupled to `dataclass` ([link to documentation](https://docs.python.org/3/library/dataclasses.html), read it!), simplifying the `tree_flatten` and `tree_unflatten` bookeeping (it is actually an excerpt of the base class I use in my JAX projects).

Take some time to study it.

> _**Note:**_ Since we're using `dataclass` to organize our class, we need to give types to its member variables. If you want to keep it simple, simply use the `jax.Array` type for JAX arrays. See [here](https://docs.jax.dev/en/latest/jax.typing.html) if you want to read more about types and JAX. See [the `jaxtyping` library](https://docs.kidger.site/jaxtyping/) if you want even more flexible typing using JAX, which we won't use in this tutorial.

In [ ]:
class PytreeDataclass:
    _dynamic_fields: ClassVar[tuple[str, ...]] = ()

    def tree_flatten(self):
        children = tuple(getattr(self, n) for n in self._dynamic_fields)

        aux = {
            'static': {f.name: getattr(self, f.name) for f in fields(self) if (f.name not in self._dynamic_fields)},
        }
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        children_kw = {name: value for name, value in zip(cls._dynamic_fields, children)}
        obj = cls(**aux['static'], **children_kw)
        return obj

We can now inherit our new classes, while registering them as a class node and dataclass:

In [ ]:
@jax.tree_util.register_pytree_node_class
@dataclass
class B(PytreeDataclass):
    myField: jax.Array
    myStaticField: str
    myDefaultField: int = field(default=2)

    _dynamic_fields = ('myField', 'myDefaultField')

    @jax.jit
    def compute(self):
        self.myField = self.myField**self.myDefaultField

Now any `B` instance is a pytree and can be manipulated as one:

In [ ]:
b = B(jnp.array([1., 2., 3.]), "Hey")

# B is a pytree, can flatten it and get its leaves and treedef
leaves, treedef = jax.tree.flatten(b)
print(f"Leaves: {leaves}")
print(f"Tree def: {treedef}")

# Manipulate leaves as any other pytree
leaves = jax.tree.map(lambda x: x**2, leaves)

# Rebuild a B object by unflattening its treedef with updated leaves
c = jax.tree.unflatten(treedef, leaves)
print(f"After pytree mapping: {c}")

# Since B is a pytree, it can also be directly manipulated
d = jax.tree.map(lambda x: x**3, b)
print(f"After direct mapping: {d}")

> _**Exercice:**_ Add some print functions in the `tree_flatten` and `tree_unflatten` methods and see when those are called.

However, jitted function still can't mutate itself:

In [ ]:
print(b.myField)
b.compute()
print(b.myField)

The reason is that a jitted method never runs on `b` at all.
To carry `b` across the jit boundary, JAX flattens it into its children, swaps those for tracers, and calls `tree_unflatten` to build a brand-new instance around them.
That reconstruction (not `b`) is the self your body executes on, so `self.myField = ...` writes into the copy.
Simply put, `b` itself is only ever read on the way in and never written.

> _**Note:**_ Add `jax.debug.print("{x}", x=hex(id(self)))` inside `compute()` next to `hex(id(b))` outside.
You will see that the identities won't match.

One solution would be to return `self` at the end of `compute()` (the copied/reconstructed object) but this can be confusing.
A clean solution is to use `dataclass.replace` ([link to the documentation](https://docs.python.org/3/library/dataclasses.html#dataclasses.replace), read it!), which imitates the JAX array behaviour:

In [ ]:
@jax.tree_util.register_pytree_node_class
@dataclass
class B(PytreeDataclass):
    myField: jax.Array
    myStaticField: str
    myDefaultField: int = field(default=2)

    _dynamic_fields = ('myField', 'myDefaultField')

    @jax.jit
    def compute(self):
        return replace(self, myField=self.myField**self.myDefaultField)

b = B(jnp.array([1., 2., 3.]), "hey")
print(b.myField)
b = b.compute()
print(b.myField)

The natural next step for `PytreeDataclass` would be an `update()` method wrapping `replace`.
But you will end up maintaining a small library while solid ones already exist.
We will introduce Equinox's `eqx.Module` next section that does exactly this.

> _**Note:**_ Returning a fresh array or module each update doesn't mean copying.
> A "new" pytree keeps the leaves it didn't change, and inside jit immutability is only semantic: once the previous array is no longer referenced, XLA reuses its buffer, so a functional update compiles down to an in-place write.
> It works exactly the same way than the `.at[idx].set()` inplace updates.

> _**Question:**_ What happens to the `_dynamic_fields` field when you inherit your class? How would you fix this? Hint: you will have to override the [`__init_subclass__` method](https://docs.python.org/3/reference/datamodel.html#object.__init_subclass__) (this is a somewhat difficult question and not really related to JAX, you might skip it).

## Using `equinox.Module`

Now that we have an idea of how to build classes compatible with the JAX ecosystem, it is time to introduce the [Equinox library](https://docs.kidger.site/equinox/).
It can basically seen as a JAX extension (with additional neural network stuff).
In particular, it adds many pytree manipulation functions and the `equinox.Module` class, which does what we tried to do with our `PytreeDataclass` tentative, but in a more mature way.

In [ ]:
class C(eqx.Module):
    myField: jax.Array
    myString: str = eqx.field(static=True)
    myDefaultField: int = eqx.field(default=2)

    @jax.jit
    def compute(self):
        return eqx.tree_at(lambda m: m.myField, self,
                           self.myField ** self.myDefaultField)

m = C(jnp.array([1., 2., 3.]), "Heeeeey")

`eqx.tree_at`([link to documentation](https://docs.kidger.site/equinox/api/manipulation/#equinox.tree_at), read it!) returns a copy of tree with selected leaves swapped, the original untouched.
Since leaf selection is a function, it can reach nested leaves or several at once (what a `dataclass.update` can't).

In [ ]:
print(m.myField)
m = m.compute()
print(m.myField)

As with `jax.Array`, and in line with JAX functionnal programming philosophy style, `eqx.Module` inherited classes are immutable.

In [ ]:
m.myField = jnp.array([10., 5., 2.])

You've now seen how more complex computations, and even whole frameworks, get built on top of JAX.
It's a lot of machinery, but don't feel you have to reach for all of it. 
Pytrees and pure functions are enough to get real work done.
The heavier tools you've seen are there for when you actually need them.
How you lay out your own code will come down to experience and taste: mine mixes pure-function collections with pytree classes, via an extended `PytreeDataclass`.